# Trening i ewaluacja detektora tablic (Kaggle, GPU)

Włodarczyk: detekcja tablic. Notebook uruchamia skrypty z repozytorium:
`train_yolo.py` -> `eval_detection.py` -> `eval_alignment.py`.

Przed uruchomieniem (prawy panel notebooka):
1. **Accelerator**: GPU T4 x2 albo P100.
2. **Internet**: włączony (klonowanie repo, wagi startowe YOLO).
3. **Add Input**: dataset `filipstanczyk/eu-license-plates-alpr`.
4. Jeśli repo jest prywatne: Add-ons -> Secrets -> `GITHUB_TOKEN` (token z dostępem do repo).

Wyniki do pobrania: `/kaggle/working/detekcja_wyniki.zip` (wagi `best.pt`, metryki CSV, wykresy).

In [ ]:
import os, subprocess
REPO = "FStanczyk/LicensePlatesN2Project"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = f"https://{token}@github.com/{REPO}.git"
except Exception:
    url = f"https://github.com/{REPO}.git"
if not os.path.exists("/kaggle/working/LicensePlatesN2Project"):
    subprocess.run(["git", "clone", "-q", url, "/kaggle/working/LicensePlatesN2Project"], check=True)
%cd /kaggle/working/LicensePlatesN2Project
!git log --oneline -1

In [ ]:
!pip install -q -e .   # pakiet alpr z zależnościami (ultralytics, ...)
DATASET = "/kaggle/input/eu-license-plates-alpr"
RUNS = "/kaggle/working/runs"
MODEL, EPOCHS, IMGSZ = "yolo11n.pt", 100, 640   # warto porównać też yolov8n.pt, yolo11s.pt i IMGSZ 960
NAME = f"{MODEL.removesuffix('.pt')}_{IMGSZ}"

## Trening (najlepsza epoka wybierana na val, wczesne zatrzymanie po 20 epokach bez poprawy)

In [ ]:
!python3 scripts/train_yolo.py --data {DATASET}/detection/data.yaml --model {MODEL} \
    --epochs {EPOCHS} --imgsz {IMGSZ} --project {RUNS} --name {NAME}

## Ewaluacja na test: P, R, F1, AP50, AP50-95 w podziale na oświetlenie, naświetlenie i zniekształcenia

In [ ]:
WEIGHTS = f"{RUNS}/{NAME}/weights/best.pt"
!python3 scripts/eval_detection.py --weights {WEIGHTS} --dataset {DATASET} --split test --imgsz {IMGSZ} --name {NAME}

## Wyrównanie perspektywy: ramki z etykiet i ramki z detektora

In [ ]:
!python3 scripts/eval_alignment.py --dataset {DATASET} --split test --examples 40 --name gt_boxes
!python3 scripts/eval_alignment.py --dataset {DATASET} --split test --preds results/detection_eval/{NAME}/predictions.csv --name {NAME}

## Wycinki z test dla OCR (Golba)

In [ ]:
!python3 scripts/detect_plates.py --weights {WEIGHTS} --dataset {DATASET} --split test --imgsz {IMGSZ}

In [ ]:
!cp {WEIGHTS} results/best_{NAME}.pt
!cp -r {RUNS}/{NAME} results/train_{NAME}
!cd /kaggle/working/LicensePlatesN2Project && zip -qr /kaggle/working/detekcja_wyniki.zip results
!ls -lh /kaggle/working/detekcja_wyniki.zip